```
# Lab type:  prompt
# Course:    NL301 Natural Language Processing with Python
# Lesson:    06 — LLM-Assisted NLP
# Task:      Complete three prompt templates for support-ticket classification
#            using the Anthropic API.
```

## Setup

Requires an `ANTHROPIC_API_KEY`:

1. Go to <a href="https://console.anthropic.com/settings/keys" target="_blank" rel="noopener noreferrer">console.anthropic.com</a> and sign in.
2. Click **Create Key** and copy it.
3. **In Colab:** open the Secrets panel (🔑 icon in the left sidebar), add a secret named `ANTHROPIC_API_KEY`, and paste your key.
   **Locally:** set it in your shell before launching Jupyter: `export ANTHROPIC_API_KEY="your-key"`.

In [1]:
!pip install anthropic --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 7.1 MB/s eta 0:00:00


In [2]:
import os

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("Anthropic API key loaded from Colab secrets.")
except ImportError:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
        if ANTHROPIC_API_KEY:
            print("Anthropic API key loaded from .env file.")
        else:
            print("Anthropic API key not found in .env file. Please ensure ANTHROPIC_API_KEY is set.")
    except ImportError:
        print("python-dotenv not installed. Please install it (`pip install python-dotenv`) or ensure ANTHROPIC_API_KEY is set as an environment variable.")
        ANTHROPIC_API_KEY = None

if ANTHROPIC_API_KEY is None:
    raise ValueError("ANTHROPIC_API_KEY is not set. Please set it in Colab secrets or a .env file.")

import anthropic
import json
from typing import Optional

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Support ticket dataset
tickets = [
    {"id": 1, "text": "My order arrived damaged and I want a full refund immediately."},
    {"id": 2, "text": "The product exceeded my expectations — absolutely fantastic!"},
    {"id": 3, "text": "Can you tell me whether you ship to Canada?"},
    {"id": 4, "text": "I've been waiting three weeks and still no delivery."},
    {"id": 5, "text": "Do you offer student discounts on annual subscriptions?"},
    {"id": 6, "text": "The app keeps crashing on iOS 17. This is unacceptable."},
    {"id": 7, "text": "Just wanted to say your support team was incredibly helpful."},
    {"id": 8, "text": "Please cancel my subscription effective immediately."},
]
CATEGORIES = ["complaint", "praise", "inquiry"]

Anthropic API key loaded from Colab secrets.


---
## Task 1: Zero-shot classification

Complete the `zero_shot_prompt` so that the model returns exactly one word: `complaint`, `praise`, or `inquiry`. No other output.

In [3]:
def classify_zero_shot(text: str) -> str:
    # TODO: write a prompt that returns exactly one category word.
    # Hint: be explicit about output format — one word, nothing else.
    zero_shot_prompt = f"""
    Classify the following support ticket into one of these categories: complaint, praise, or inquiry.
    Return only the category word, nothing else.

    Ticket: {text}
    """

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=16,
        messages=[{"role": "user", "content": zero_shot_prompt}],
    )
    return response.content[0].text.strip().lower()

# Test on first 4 tickets
for t in tickets[:4]:
    label = classify_zero_shot(t["text"])
    print(f"[{label:10s}]  {t['text'][:60]}")

[complaint ]  My order arrived damaged and I want a full refund immediatel
[praise    ]  The product exceeded my expectations — absolutely fantastic!
[inquiry   ]  Can you tell me whether you ship to Canada?
[complaint ]  I've been waiting three weeks and still no delivery.


**Analysis:** Does zero-shot correctly classify the complaint about a damaged order? The cancellation request — is that a complaint or inquiry?

*(Write your answer here.)*

<details>
<summary>🔑 Model prompt — Task 1</summary>

**Example strong prompt:**

> Classify the following support ticket into exactly one category.
>
> Categories:
> - complaint: expresses dissatisfaction, reports a problem, or requests compensation/cancellation
> - praise: expresses satisfaction or appreciation
> - inquiry: asks a question or requests information
>
> Reply with only the single category word (complaint, praise, or inquiry). No explanation, no punctuation, nothing else.
>
> Ticket: {text}

**Why it's strong:** Defines each category with a brief criterion (resolves the complaint/inquiry boundary for cancellation requests), specifies `max_tokens=16` to prevent verbose output, and ends with "nothing else" to suppress explanations the model might add by default.

**Analysis:** Zero-shot correctly classifies the damaged order (complaint). The cancellation request (ticket 8) is ambiguous — "Please cancel my subscription" is a service request, so either `complaint` or `inquiry` can be argued; the category definitions above map it to `inquiry` (requesting an action/information) unless the tone implies dissatisfaction.

</details>

---
## Task 2: Few-shot classification

Add 2 labelled examples per category to your prompt. Then compare zero-shot vs few-shot accuracy on all 8 tickets (use manual ground-truth labels below).

In [4]:
GROUND_TRUTH = {1: "complaint", 2: "praise", 3: "inquiry", 4: "complaint",
                5: "inquiry",   6: "complaint", 7: "praise", 8: "inquiry"}

def classify_few_shot(text: str) -> str:
    # TODO: build a few-shot prompt with 2 examples per category (6 examples total).
    # Each example: show the ticket text and the correct label.
    few_shot_prompt = f"""
    Classify the following support ticket into one of these categories: complaint, praise, or inquiry.
    Return only the category word, nothing else.

    Ticket: My order arrived damaged and I want a full refund immediately.
    complaint

    Ticket: I've been waiting three weeks and still no delivery.
    complaint

    Ticket: The product exceeded my expectations — absolutely fantastic!
    praise

    Ticket: Just wanted to say your support team was incredibly helpful.
    praise

    Ticket: Can you tell me whether you ship to Canada?
    inquiry

    Ticket: Do you offer student discounts on annual subscriptions?
    inquiry

    Ticket: {text}
    """

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=16,
        messages=[{"role": "user", "content": few_shot_prompt}],
    )
    return response.content[0].text.strip().lower()

# Evaluate both approaches
def accuracy(fn):
    correct = sum(fn(t["text"]) == GROUND_TRUTH[t["id"]] for t in tickets)
    return correct / len(tickets)

print(f"Zero-shot accuracy: {accuracy(classify_zero_shot):.2f}")
print(f"Few-shot  accuracy: {accuracy(classify_few_shot):.2f}")

Zero-shot accuracy: 0.88
Few-shot  accuracy: 0.88


**Analysis:** Which approach performed better? When would you expect few-shot to consistently outperform zero-shot?

*(Write your answer here.)*

<details>
<summary>🔑 Model prompt — Task 2</summary>

**Example strong few-shot prompt (2 examples per category):**

> Classify the support ticket below into: complaint, praise, or inquiry. Reply with one word only.
>
> Ticket: My order arrived damaged and I want a full refund immediately.
> Category: complaint
>
> Ticket: I've been waiting three weeks and still no delivery.
> Category: complaint
>
> Ticket: The product exceeded my expectations — absolutely fantastic!
> Category: praise
>
> Ticket: Just wanted to say your support team was incredibly helpful.
> Category: praise
>
> Ticket: Can you tell me whether you ship to Canada?
> Category: inquiry
>
> Ticket: Do you offer student discounts on annual subscriptions?
> Category: inquiry
>
> Ticket: {text}
> Category:

**Why it's strong:** Consistent `Ticket:` / `Category:` structure allows the model to pattern-match and append only the label. Leaving `Category:` as a prompt suffix almost eliminates explanatory output. Two examples per class cover the complaint/inquiry boundary edge case.

**Analysis:** Few-shot typically matches or slightly outperforms zero-shot on well-defined tasks with a small label set. You would expect few-shot to consistently outperform zero-shot when: (a) category boundaries are ambiguous in natural language, (b) the domain uses non-standard terminology the model hasn't seen, or (c) the output format needs strict enforcement (e.g., a single word, a JSON schema).

</details>

---
## Task 3: Structured JSON extraction

Write a prompt that extracts four fields from each ticket: `sentiment` (positive/negative/neutral), `issue_type` (billing/shipping/technical/general), `urgency` (low/medium/high), `action_required` (bool). Handle `json.JSONDecodeError`.

In [6]:
from dataclasses import dataclass
import re

@dataclass
class TicketAnalysis:
    sentiment: str
    issue_type: str
    urgency: str
    action_required: bool

def extract_ticket_info(text: str) -> Optional[TicketAnalysis]:
    extraction_prompt = f"""
    Extract the following information from the support ticket provided below, and return it as a JSON object.
    Only return the JSON object, nothing else.

    Fields to extract:
    - `sentiment`: overall sentiment of the ticket (options: "positive", "negative", "neutral")
    - `issue_type`: the main type of issue (options: "billing", "shipping", "technical", "general")
    - `urgency`: how urgent the issue is (options: "low", "medium", "high")
    - `action_required`: a boolean indicating if an action is required by support (true/false)

    Ticket: {text}
    """

    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": extraction_prompt}],
        )
        # Extract JSON string from potentially markdown-formatted response
        json_match = re.search(r'```json\n(.*)\n```', response.content[0].text.strip(), re.DOTALL)
        if json_match:
            json_string = json_match.group(1)
        else:
            json_string = response.content[0].text.strip()

        data = json.loads(json_string)
        return TicketAnalysis(**data)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        print(f"Raw response: {response.content[0].text[:200]}")
        return None

for t in tickets[:3]:
    result = extract_ticket_info(t["text"])
    print(f"Ticket {t['id']}: {result}")

Ticket 1: TicketAnalysis(sentiment='negative', issue_type='shipping', urgency='high', action_required=True)
Ticket 2: TicketAnalysis(sentiment='positive', issue_type='general', urgency='low', action_required=False)
Ticket 3: TicketAnalysis(sentiment='neutral', issue_type='general', urgency='low', action_required=True)


**Analysis questions:**

1. At roughly 200 input tokens + 50 output tokens per ticket, and Claude Haiku pricing (~$0.80 per 1M input tokens, ~$4 per 1M output tokens), what is the approximate daily API cost for 10,000 tickets?
2. Under what conditions would few-shot reliably outperform zero-shot for classification tasks like this?
3. In a production pipeline, beyond `JSONDecodeError`, what other error scenarios should you handle?

*(Write your answers here.)*

<details>
<summary>🔑 Reveal answers — Task 3 analysis</summary>

**1. Approximate daily API cost for 10,000 tickets:**

- Input: 200 tokens × 10,000 = 2M tokens × $0.80/1M = **$1.60/day**
- Output: 50 tokens × 10,000 = 0.5M tokens × $4.00/1M = **$2.00/day**
- Total: **~$3.60/day** at Claude Haiku pricing.

**2. When few-shot reliably outperforms zero-shot:**

Few-shot is most valuable when category boundaries are ambiguous (e.g., a cancellation request that is both a complaint and an action request), when the domain vocabulary differs from general text, or when the required output format needs strict enforcement. For simple, well-separated categories on standard vocabulary, zero-shot often matches few-shot — the cost of additional tokens in the prompt may not be worth it.

**3. Production error handling beyond `JSONDecodeError`:**

- `anthropic.RateLimitError` — implement exponential back-off with jitter
- `anthropic.APITimeoutError` / network errors — retry with timeout ceiling
- Missing or extra JSON fields — validate against a schema (e.g., `pydantic`) before constructing `TicketAnalysis`
- Invalid enum values (e.g., `urgency: "critical"` not in allowed set) — normalise or reject and log
- Empty or `None` response content — guard before `response.content[0].text`
- Persistent failures — route to a dead-letter queue for human review

</details>